# Prototype recurrence across downstream datasets

Prototypes are learned once over HEEDB during pretraining. Each downstream dataset
(PTB-XL, CinC, CODE-15%) then independently *assigns* a fixed number of those
prototypes to each of its labels via the ILP step. A prototype that several datasets
pick for the same clinical concept is **recurrent**: the same HEEDB waveform is doing
the same work everywhere, which is the evidence that the prototype captures the
concept rather than a dataset artifact.

This notebook generalizes the single-label (RBBB) version in `proto-recur.ipynb` to
an arbitrary set of labels and produces:

1. per-label overlap tables and Venn diagrams of the assigned prototype sets,
2. the list of recurrent prototypes per label,
3. clinical printouts of a chosen prototype as projected by every run that uses it
   (HEEDB pretraining, plus each downstream dataset's probe and fine-tuned models).

Everything below is driven by `LABELS` (the label mapping) and `DATASETS` (the run
registry), so adding a label or a dataset only means editing those two constants.

In [ ]:
import itertools
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from protossl.datasets import (
    CincECGDataset,
    Code15ECGDataset,
    HeedbECGDataset,
    PtbxlECGDataset,
)
from protossl.defines import (
    CINC_TARGETS,
    CODE15_TARGETS,
    HEEDB_TARGETS,
    PTBXL_TARGETS,
)
from protossl.plotting import plot_ecg, resolve_index

# runs
RUN_DIR = Path("/opt/gpu_working/steven/protossl-ecg-outputs")
PRETRAIN_DIR = RUN_DIR / "protossl-heedb"  # HEEDB pretraining, source of all prototypes
EXPERIMENT_DIR = RUN_DIR / "experiments-seed42"  # downstream runs, one dir per dataset
MODEL = "protossl-heedb-pila"  # linear probe run, "-ft" suffix is the fine-tuned run

# data
DATA_DIR = Path("/opt/gpudata/ecg")
SAMPLING_RATE = 100  # models were trained at 100 Hz
SPLIT = "train"  # prototypes are projected onto the train split (see protossl.trainer)

# model, see configs/target-guided-14ppl.yaml
PPL = 14  # prototypes per label (n_prototypes_per_label)

FIG_DIR = Path("figs")
FIG_DIR.mkdir(parents=True, exist_ok=True)

## Datasets and labels

`DATASETS` holds the three downstream datasets that assign prototypes; HEEDB is kept
separate in `PRETRAIN` because it is the source of the prototypes, not a consumer of
them (its pretraining is unsupervised, so it never assigns prototypes to labels).

`LABELS` maps a canonical label name onto the exact target string used by each
dataset's label set. Edit the right-hand sides to change the clinical mapping; the
next cell checks every string against the corresponding `*_TARGETS` in
`protossl.defines`. Two mappings are judgement calls worth knowing about:

- **RBBB**: PTB-XL and CinC distinguish complete from incomplete RBBB, so the mapping
  uses `CRBBB`. CinC also carries a plain `RBBB` code; swap it in if the looser
  definition is wanted (`IRBBB` is the incomplete-only code in both).
- **LBBB**: PTB-XL again splits complete/incomplete, so the mapping uses `CLBBB`.
  CinC has only `LBBB`/`ILBBB`, so `LBBB` there already means complete.

In [ ]:
PRETRAIN = {
    "slug": "heedb",
    "dataset_cls": HeedbECGDataset,
    "dataset_path": DATA_DIR / "heedb",
    "targets": list(HEEDB_TARGETS),  # a dict of label -> statement codes
}

DATASETS = {
    "PTB-XL": {
        "slug": "ptbxl",
        "run_dir": EXPERIMENT_DIR / "runs-ptbxl",
        "dataset_cls": PtbxlECGDataset,
        "dataset_path": DATA_DIR / "ptb-xl",
        "targets": PTBXL_TARGETS,
    },
    "CinC": {
        "slug": "cinc",
        "run_dir": EXPERIMENT_DIR / "runs-cinc",
        "dataset_cls": CincECGDataset,
        "dataset_path": DATA_DIR / "cinc-2020",
        "targets": CINC_TARGETS,
    },
    "CODE-15%": {
        "slug": "code15",
        "run_dir": EXPERIMENT_DIR / "runs-code15",
        "dataset_cls": Code15ECGDataset,
        "dataset_path": DATA_DIR / "code15",
        "targets": CODE15_TARGETS,
    },
}

# canonical label -> that label's exact target string in each label set
LABELS = {
    "1st-degree AV block": {
        "HEEDB": "WITH 1ST DEGREE AV BLOCK",
        "PTB-XL": "1AVB",
        "CinC": "IAVB",
        "CODE-15%": "1dAVb",
    },
    "RBBB": {
        "HEEDB": "RIGHT BUNDLE BRANCH BLOCK",
        "PTB-XL": "CRBBB",
        "CinC": "CRBBB",
        "CODE-15%": "RBBB",
    },
    "LBBB": {
        "HEEDB": "LEFT BUNDLE BRANCH BLOCK",
        "PTB-XL": "CLBBB",
        "CinC": "LBBB",
        "CODE-15%": "LBBB",
    },
    "AFib": {
        "HEEDB": "ATRIAL FIBRILLATION",
        "PTB-XL": "AFIB",
        "CinC": "AF",
        "CODE-15%": "AF",
    },
    "Sinus Brady": {
        "HEEDB": "SINUS BRADYCARDIA",
        "PTB-XL": "SBRAD",
        "CinC": "SB",
        "CODE-15%": "SB",
    },
    "Sinus Tach": {
        "HEEDB": "SINUS TACHYCARDIA",
        "PTB-XL": "STACH",
        "CinC": "STach",
        "CODE-15%": "ST",
    },
}

# filename-safe stems for saved figures
LABEL_SLUGS = {
    "1st-degree AV block": "1davb",
    "RBBB": "rbbb",
    "LBBB": "lbbb",
    "AFib": "afib",
    "Sinus Brady": "sbrad",
    "Sinus Tach": "stach",
}

In [ ]:
# every mapped name must exist in the label set it refers to, otherwise a typo
# silently turns into an empty prototype set downstream
for label, mapping in LABELS.items():
    assert label in LABEL_SLUGS, f"no filename slug for {label}"
    assert mapping["HEEDB"] in PRETRAIN["targets"], f"{label}: bad HEEDB target"
    for name, cfg in DATASETS.items():
        assert mapping[name] in cfg["targets"], f"{label}: bad {name} target"

pd.DataFrame(LABELS).T

## Load assignment and projection metadata

- `assignment_metadata.csv` (one row per label slot) says which pretrained prototype
  the ILP gave to each label slot of a downstream dataset.
- `projection_metadata.csv` (one row per prototype) says which sample and 1-second
  window each prototype was projected onto, i.e. the waveform to plot.

In [ ]:
def read_metadata(path: Path) -> pd.DataFrame:
    """Read a metadata CSV, normalizing the sample ID column name.

    `ecg_id` was renamed to `sample_id` when the datasets were generalized beyond
    ECG (see `protossl.trainer.PredictionWriter`), so runs written before that
    rename still carry the old column name.
    """
    return pd.read_csv(path).rename(columns={"ecg_id": "sample_id"})


heedb_proj = read_metadata(
    PRETRAIN_DIR / "project-prototypes/latest/projection_metadata.csv"
)

for name, cfg in DATASETS.items():
    run = cfg["run_dir"]
    cfg["assign"] = read_metadata(
        run / MODEL / "learn-prototype-assignments/latest/assignment_metadata.csv"
    )
    cfg["proj"] = read_metadata(
        run / MODEL / "project-prototypes-supervised/latest/projection_metadata.csv"
    )
    cfg["ft_proj"] = read_metadata(
        run
        / f"{MODEL}-ft"
        / "project-prototypes-supervised/latest/projection_metadata.csv"
    )

In [ ]:
# the row position of a prototype in `proj`/`ft_proj` is its slot in the projected
# model, and assignment rows are written label-major then slot-minor, so an
# assignment row's position is exactly that slot. the checks below pin down that
# correspondence, which is what lets us index projections by assignment row.
assert (heedb_proj["prototype_id"] == heedb_proj.index).all()

for name, cfg in DATASETS.items():
    assign, proj, ft_proj = cfg["assign"], cfg["proj"], cfg["ft_proj"]
    n_slots = len(cfg["targets"]) * PPL
    assert len(assign) == len(proj) == len(ft_proj) == n_slots, name
    assert (assign.index == assign["label_idx"] * PPL + assign["slot_idx"]).all(), name
    assert (proj["prototype_id"] == proj.index).all(), name
    assert (ft_proj["prototype_id"] == ft_proj.index).all(), name
    # prototypes come from the pretrained bank, which HEEDB projection enumerates
    assert assign["prototype_idx"].isin(heedb_proj.index).all(), name

print(f"{len(heedb_proj)} pretrained prototypes")

## Prototypes assigned to each label

The ILP (`ilp_effect_size`) uses each prototype at most once per dataset, so within a
dataset the prototype sets of different labels are disjoint, and each label gets
exactly `PPL` prototypes.

In [ ]:
def label_prototypes(label: str) -> dict[str, set[int]]:
    """Pretrained prototype indices each dataset assigned to `label`."""
    sets = {}
    for name, cfg in DATASETS.items():
        target = LABELS[label][name]
        rows = cfg["assign"].loc[cfg["assign"]["label_name"] == target]
        assert len(rows) == PPL, f"{name} {target}: {len(rows)} slots != {PPL}"
        sets[name] = set(rows["prototype_idx"])
    return sets


PROTOTYPES = {label: label_prototypes(label) for label in LABELS}

## Overlap between datasets

`region_table` splits the prototypes into the mutually exclusive regions of a Venn
diagram: `"PTB-XL only"` counts prototypes PTB-XL assigned to the label and no one
else, `"PTB-XL & CinC"` counts prototypes both of them (but not CODE-15%) assigned,
and so on.

In [ ]:
def region_table(sets: dict[str, set[int]]) -> pd.DataFrame:
    """Break `sets` into the mutually exclusive regions of a Venn diagram.

    Every region is listed, including empty ones, so that tables for different
    labels line up. Regions are ordered by how many datasets share them.
    """
    names = list(sets)
    rows = []
    for size in range(1, len(names) + 1):
        for combo in itertools.combinations(names, size):
            shared = set.intersection(*(sets[n] for n in combo))
            others = [sets[n] for n in names if n not in combo]
            members = shared.difference(*others) if others else shared
            rows.append(
                {
                    "region": " & ".join(combo) + (" only" if size == 1 else ""),
                    "n_datasets": size,
                    "count": len(members),
                    "prototypes": sorted(members),
                }
            )
    out = pd.DataFrame(rows)
    assert out["count"].sum() == len(set().union(*sets.values()))
    return out


def add_total(table: pd.DataFrame) -> pd.DataFrame:
    """Append a total row, for display only."""
    total = {"region": "Total", "n_datasets": None, "count": table["count"].sum()}
    return pd.concat([table, pd.DataFrame([total])], ignore_index=True)

In [ ]:
LABEL = "RBBB"  # label to look at in detail below

add_total(region_table(PROTOTYPES[LABEL]))[["region", "count", "prototypes"]]

In [ ]:
# every label at once: rows are Venn regions, columns are labels
overlap = pd.DataFrame(
    {
        label: region_table(sets).set_index("region")["count"]
        for label, sets in PROTOTYPES.items()
    }
)
overlap.loc["Total"] = [
    len(set().union(*sets.values())) for sets in PROTOTYPES.values()
]
overlap.rename_axis("Datasets")

In [ ]:
# share of a label's prototypes that at least two datasets agreed on
shared = pd.DataFrame(
    {
        label: region_table(sets).groupby("n_datasets")["count"].sum()
        for label, sets in PROTOTYPES.items()
    }
).T
(shared[[c for c in shared.columns if c > 1]].sum(axis=1) / shared.sum(axis=1)).rename(
    "recurrent_fraction"
).to_frame()

In [ ]:
# !pip install matplotlib-venn
from matplotlib_venn import venn3

assert len(DATASETS) == 3, "venn3 draws exactly three sets"

n_cols = 3
n_rows = -(-len(PROTOTYPES) // n_cols)  # ceiling division
fig, axs = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
for ax, (label, sets) in zip(axs.flat, PROTOTYPES.items()):
    venn3([sets[name] for name in DATASETS], set_labels=tuple(DATASETS), ax=ax)
    ax.set_title(label)
for ax in axs.flat[len(PROTOTYPES) :]:  # blank out any unused panels
    ax.axis("off")
fig.tight_layout()
fig.savefig(FIG_DIR / "proto-recur-venn.png", dpi=300, bbox_inches="tight")

## Recurrent prototypes

A prototype is recurrent for a label when at least `min_datasets` datasets picked it
for that label.

In [ ]:
def recurrent_prototypes(label: str, min_datasets: int = 2) -> list[int]:
    """Prototypes assigned to `label` by at least `min_datasets` datasets."""
    table = region_table(PROTOTYPES[label])
    hits = table.loc[table["n_datasets"] >= min_datasets, "prototypes"]
    return sorted(p for members in hits for p in members)


recurrent = {label: recurrent_prototypes(label) for label in LABELS}
pd.Series(recurrent).rename("prototypes").to_frame().assign(
    count=lambda df: df["prototypes"].str.len()
)

## Plot a prototype's projections

Each run projects a prototype onto whichever training sample and 1-second window
activates it most, so the same prototype has a different source waveform per run.
Comparing those waveforms is what shows the concept surviving across datasets.

In [ ]:
def prototype_sources(proto_idx: int) -> pd.DataFrame:
    """Sample and window each run projected prototype `proto_idx` onto.

    Includes the HEEDB pretraining projection plus, for every dataset that assigned
    the prototype, its linear probe and fine-tuned projections.
    """
    rows = [{"dataset": "HEEDB", "run": "pretrain", **heedb_proj.loc[proto_idx]}]
    for name, cfg in DATASETS.items():
        slots = cfg["assign"].index[cfg["assign"]["prototype_idx"] == proto_idx]
        if len(slots) == 0:
            continue  # this dataset did not use the prototype for any label
        # ilp_effect_size uses a prototype at most once per dataset
        assert len(slots) == 1, f"{name}: prototype {proto_idx} used {len(slots)} times"
        slot = slots[0]
        rows.append({"dataset": name, "run": "probe", **cfg["proj"].loc[slot]})
        rows.append({"dataset": name, "run": "ft", **cfg["ft_proj"].loc[slot]})
    columns = ["dataset", "run", "source_id", "sample_id", "chunk_idx", "emb_sim"]
    ids = ["source_id", "sample_id", "chunk_idx"]  # floated by the mixed-dtype rows
    return pd.DataFrame(rows)[columns].astype(dict.fromkeys(ids, int))

In [ ]:
_datasets = {**DATASETS, "HEEDB": PRETRAIN}
_dataset_cache: dict[tuple[str, str], object] = {}


def get_dataset(name: str, split: str = SPLIT):
    """Build a dataset once and reuse it, loading waveforms is expensive."""
    if (name, split) not in _dataset_cache:
        cfg = _datasets[name]
        _dataset_cache[(name, split)] = cfg["dataset_cls"](
            dataset_path=str(cfg["dataset_path"]),
            split=split,
            sampling_rate=SAMPLING_RATE,
        )
    return _dataset_cache[(name, split)]


def plot_prototype(
    label: str,
    proto_idx: int,
    split: str = SPLIT,
    save: bool = True,
) -> dict[str, plt.Figure]:
    """Plot the source window of `proto_idx` for every run that uses it."""
    figs = {}
    for _, row in prototype_sources(proto_idx).iterrows():
        stem = (
            f"{LABEL_SLUGS[label]}-p{proto_idx}"
            f"-{_datasets[row['dataset']]['slug']}-{row['run']}"
        )
        figs[stem] = plot_ecg(
            dataset=get_dataset(row["dataset"], split=split),
            sample_id=int(row["sample_id"]),
            chunk_idx=int(row["chunk_idx"]),
            title=f"{label} | prototype {proto_idx} | {row['dataset']} ({row['run']})",
        )
        if save:
            figs[stem].savefig(FIG_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
    return figs

In [ ]:
# prefer a prototype every dataset agreed on, then any recurrent one, then any at all
candidates = recurrent_prototypes(LABEL, min_datasets=len(DATASETS))
if not candidates:
    candidates = recurrent_prototypes(LABEL)
if not candidates:
    candidates = sorted(set().union(*PROTOTYPES[LABEL].values()))

PROTO_IDX = candidates[0]  # or any prototype index of interest

prototype_sources(PROTO_IDX)

In [ ]:
figs = plot_prototype(LABEL, PROTO_IDX)
list(figs)

## Optional: HEEDB labels of the prototype's source ECG

The pretraining is unsupervised, so a prototype has no HEEDB label of its own. What
it does have is a source ECG, and that ECG's HEEDB statements are a useful sanity
check on whether the prototype really encodes the concept the downstream datasets
assigned it to. Set `USE_HEEDB_150=1` before importing `protossl` if the pretraining
run used the 150-label HEEDB set, otherwise these names come from the 20-label set.

In [ ]:
def heedb_labels(sample_id: int, split: str = SPLIT) -> list[str]:
    """HEEDB statements carried by a source ECG."""
    ds = get_dataset("HEEDB", split=split)
    idx = resolve_index(ds, sample_id=sample_id)
    return [
        name
        for name, present in zip(PRETRAIN["targets"], ds.labels[idx].tolist())
        if present
    ]


expected = LABELS[LABEL]["HEEDB"]
ecg_id = int(heedb_proj.loc[PROTO_IDX, "sample_id"])
statements = heedb_labels(ecg_id)
print(f"prototype {PROTO_IDX} came from HEEDB ECG {ecg_id}")
seen = "present" if expected in statements else "absent"
print(f"expected statement {expected}: {seen}")
statements